<a href="https://colab.research.google.com/github/ankitarchoudhary/IBM-Data-Science-Capstone/blob/main/Phase1_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Data Collection - SpaceX Falcon 9 Capstone


This phase collects data from two sources:
1. **SpaceX REST API** - structured launch data (JSON format)
2. **Wikipedia web scraping** - additional details not available in the API

Note: The live SpaceX API was temporarily unavailable (HTTP 525 error, a server-side outage),
so this version uses IBM's official fallback dataset for Part A, which contains the exact same
columns the API would normally produce. The API code is still included below (commented out)
in case you want to try it again later once the API is back online.


## Part A: SpaceX Launch Data

First, import the libraries needed for this section.

In [ ]:
import pandas as pd
import numpy as np
import requests
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

**Explanation:**
- `pandas` handles tabular data (rows and columns), similar to an Excel sheet.
- `numpy` provides numerical operations, such as computing a mean for missing values.
- `requests` sends HTTP requests to APIs and websites.
- `datetime` handles date values, such as filtering by a cutoff date.
- The two `pd.set_option` lines just make printed tables easier to read by not truncating columns.

### Attempting the live SpaceX API

The cell below attempts to fetch live data from the SpaceX REST API. If the API is down
(for example, returning a 525 or other error status), this cell will raise an error - that
is expected and not a mistake in the code. Run it to check the API's current status.

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches"
response = requests.get(spacex_url)
print("Status code:", response.status_code)

Status code: 525


**Explanation:**
- `spacex_url` stores the address of the SpaceX API endpoint that returns all launches.
- `requests.get(spacex_url)` sends a GET request (a request to retrieve data, not modify anything).
- `response.status_code` reports how the server handled the request. **200 means success.**
  Any other code (404, 500, 525, etc.) means something went wrong on the server's side.

In [ ]:
if response.status_code == 200:
    data = response.json()
    data = pd.json_normalize(data)
    print("Live API data loaded successfully.")
    print(data.shape)
else:
    print("API is not returning data right now (status code:", response.status_code, ")")
    print("Using the IBM-provided fallback dataset instead (see next section).")

API is not returning data right now (status code: 525 )
Using the IBM-provided fallback dataset instead (see next section).


**Explanation:**
- This checks the status code before trying to parse the response as JSON. Trying to call
  `.json()` on a failed response (like the 525 error) raises a `JSONDecodeError`, because the
  server sends back an HTML error page instead of JSON data - that is exactly the error you saw earlier.
- `pd.json_normalize(data)` flattens the nested JSON structure into a proper table, only run
  if the request succeeded.

### Fallback: IBM's Official Dataset

IBM provides this exact dataset (with the same columns the API call above would produce) as an
official fallback, specifically for situations where the live API is temporarily unavailable.
This is not a workaround or a shortcut - it is the dataset the course itself points students to
when the API has connectivity issues.

In [ ]:
fallback_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"

try:
    data_falcon9 = pd.read_csv(fallback_url)
    print("Fallback dataset loaded from IBM's server.")
except Exception as e:
    print("Could not reach IBM's server either:", e)
    print("Upload 'dataset_part_1_fallback.csv' manually using the Files panel on the left,")
    print("then run: data_falcon9 = pd.read_csv('dataset_part_1_fallback.csv')")

data_falcon9.head()

Fallback dataset loaded from IBM's server.


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


**Explanation:**
- `fallback_url` points to IBM's own hosted copy of this dataset.
- `pd.read_csv(fallback_url)` reads a CSV file directly from a URL into a DataFrame.
- The `try/except` block handles the case where even this URL is unreachable (for example, if
  Colab's network has an issue): it falls back to reading a local file instead, which you would
  need to upload manually to Colab's Files panel first.
- `data_falcon9.head()` displays the first 5 rows so you can confirm the data loaded correctly.

In [ ]:
data_falcon9.shape

(90, 18)

**Explanation:** `.shape` returns (number of rows, number of columns), confirming how much data was loaded.

In [ ]:
data_falcon9.dtypes

,0
FlightNumber,int64
Date,object
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


**Explanation:** `.dtypes` shows the data type of each column (e.g. `int64` for whole numbers,
`float64` for decimals, `object`/`str` for text). This is useful to confirm columns are stored
in the type you expect before doing calculations on them.

In [ ]:
data_falcon9.isnull().sum()

,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


**Explanation:** `.isnull().sum()` counts missing values in each column. In this dataset,
`LandingPad` is expected to have missing values for launches where no landing pad was used
(for example, ocean landings or failed attempts) - that is normal, not an error.

In [ ]:
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean())

data_falcon9.isnull().sum()

,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


**Explanation:**
- `.fillna(data_falcon9['PayloadMass'].mean())` replaces any missing `PayloadMass` values with
  the average (mean) of that column. This is a simple, common way to handle missing numeric data
  without having to drop rows entirely.
- Re-running `.isnull().sum()` confirms that `PayloadMass` no longer has missing values.

In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Saved: dataset_part_1.csv")

Saved: dataset_part_1.csv


**Explanation:**
- `.to_csv('dataset_part_1.csv', index=False)` saves the cleaned DataFrame to a CSV file, so it
  can be reloaded in later phases (data wrangling, EDA) without repeating this step.
- `index=False` means the row numbers (0, 1, 2, ...) are not saved as an extra column.
- In Colab, this file will appear in the left-side "Files" panel - right-click it and choose
  **Download** to save it locally.

## Part B: Wikipedia Web Scraping

This section scrapes the "List of Falcon 9 and Falcon Heavy launches" Wikipedia page for
additional columns (such as Customer) that are not available from the API.

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(static_url, headers=headers)
print(response.status_code)

200


**Explanation:**
- `BeautifulSoup` is a library that parses HTML (a webpage's underlying code) so that specific
  elements, such as tables, can be located and extracted.
- `static_url` points to a **fixed, archived version** of the Wikipedia page (`oldid=...`),
  rather than the live page. This matters because the live page is updated continuously as new
  launches happen; using a fixed version keeps the results consistent and reproducible.
- The rest of the cell sends the request and checks the status code, as before.

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title)

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


**Explanation:**
- `BeautifulSoup(response.text, 'html.parser')` converts the raw HTML text into a `soup` object
  that can be searched and navigated easily.
- `soup.title` prints the page title, just to confirm the correct page was retrieved.

In [ ]:
html_tables = soup.find_all('table')
print("Total tables found:", len(html_tables))

first_launch_table = html_tables[2]
print(first_launch_table.prettify()[:500])

Total tables found: 25
<table class="wikitable plainrowheaders collapsible" id="mwmQ" style="width: 100%;">
 <tbody id="mwmg">
  <tr id="mwmw">
   <th id="mwnA" scope="col">
    Flight No.
   </th>
   <th id="mwnQ" scope="col">
    Date and
    <br id="mwng"/>
    time (
    <a href="//en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwnw" rel="mw:WikiLink" title="Coordinated Universal Time">
     UTC
    </a>
    )
   </th>
   <th id="mwoA" scope="col">
    <a href="//en.wikipedia.org/wiki/List_of_Falcon_9_first


**Explanation:**
- `soup.find_all('table')` finds every `<table>` element on the page and returns them as a list.
- `html_tables[2]` selects the third table (index 2, since counting starts at 0) - on this
  specific Wikipedia page, that is the table containing the actual launch records (the first two
  tables are used for other content, such as summary statistics).
- `.prettify()[:500]` prints a nicely formatted preview of the table's HTML, limited to the first
  500 characters, just to inspect its structure.

In [ ]:
column_names = []

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not column_name.strip().isdigit():
        column_name = column_name.strip()
        return column_name

temp = first_launch_table.find_all('th')
for th in temp:
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


**Explanation:**
- `extract_column_from_header` is a helper function that extracts a clean column name from a
  table header cell (`<th>`).
- `row.br.extract()`, `row.a.extract()`, `row.sup.extract()` remove line breaks, links, and
  footnote superscripts (like `[1]`) that are sometimes embedded in headers, leaving only plain text.
- `' '.join(row.contents)` joins the remaining pieces into a single clean string.
- `if not column_name.strip().isdigit()` skips headers that are just numbers (used for row
  numbering), since those should not be treated as column names.
- `first_launch_table.find_all('th')` finds every header cell in the table, and the loop extracts
  a name from each one, appending it to `column_names`.

In [ ]:
launch_dict2 = dict.fromkeys(column_names)

if 'Date and time ( )' in launch_dict2:
    del launch_dict2['Date and time ( )']

launch_dict2['Flight No.'] = []
launch_dict2['Launch site'] = []
launch_dict2['Payload'] = []
launch_dict2['Payload mass'] = []
launch_dict2['Orbit'] = []
launch_dict2['Customer'] = []
launch_dict2['Launch outcome'] = []
launch_dict2['Version Booster'] = []
launch_dict2['Booster landing'] = []
launch_dict2['Date'] = []
launch_dict2['Time'] = []

print(list(launch_dict2.keys()))

['Flight No.', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Version Booster', 'Booster landing', 'Date', 'Time']


**Explanation:**
- `dict.fromkeys(column_names)` creates a dictionary where each column name becomes a key, with
  all values initially set to `None`.
- The 'Date and time' column is removed because it will be manually split into separate 'Date'
  and 'Time' columns below.
- The remaining lines add new keys that will be filled in row by row in the next step.
- Note: if the exact column name printed here differs slightly, this line may need adjusting -
  check the printed output and flag it if something looks off.

In [ ]:
import re

extracted_row = 0

for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        else:
            flag = False

        row = rows.find_all('td')

        if flag:
            extracted_row += 1
            launch_dict2['Flight No.'].append(flight_number)

            raw_datetime = row[0].get_text(strip=True)
            match = re.match(r'(\d{1,2}\s+\w+\s+\d{4})[,]?\s*(\d{1,2}:\d{2}(?::\d{2})?)', raw_datetime)
            if match:
                launch_dict2['Date'].append(match.group(1))
                launch_dict2['Time'].append(match.group(2))
            else:
                launch_dict2['Date'].append(raw_datetime)
                launch_dict2['Time'].append(None)

            launch_dict2['Version Booster'].append(row[1].get_text(strip=True))
            launch_dict2['Launch site'].append(row[2].get_text(strip=True))
            launch_dict2['Payload'].append(row[3].get_text(strip=True))
            launch_dict2['Payload mass'].append(row[4].get_text(strip=True))
            launch_dict2['Orbit'].append(row[5].get_text(strip=True))
            launch_dict2['Customer'].append(row[6].get_text(strip=True))
            launch_dict2['Launch outcome'].append(row[7].get_text(strip=True))
            launch_dict2['Booster landing'].append(row[8].get_text(strip=True) if len(row) > 8 else None)

print("Total rows extracted:", extracted_row)

Total rows extracted: 121


**Explanation (this is the main extraction loop):**
- The outer loop finds every table with the class `"wikitable plainrowheaders collapsible"` -
  the standard class Wikipedia uses for these launch tables, which helps target the right tables.
- The inner loop goes through every row (`<tr>`) in each table.
- `if rows.th and rows.th.string` checks whether the row's first cell is a number (the Flight
  No.); only rows that pass this check are treated as actual launch records (this skips
  section-header rows).
- `row = rows.find_all('td')` extracts all data cells in that row.
- Each `row[i].get_text(strip=True)` extracts the text from a specific column and appends it to
  the dictionary - for example, `row[1]` is the booster version, `row[2]` is the launch site, and
  so on, based on the table's column order.
- **Important:** Wikipedia's table structure can occasionally shift slightly. If any column
  looks misaligned in the output, the `row[i]` indices may need adjusting - check the output and
  flag anything that looks wrong.

In [ ]:
df2 = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict2.items() if len(value) > 0})
df2.head()

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,"CCAFS,SLC-40",Dragon Spacecraft Qualification Unit,,LEO,SpaceX,Success,F9 v1.0[7]B0003.1[8],Failure[9][10](parachute),4 June 2010,18:45
1,2,"CCAFS,SLC-40",Dragondemo flight C1(Dragon C101),,LEO(ISS),NASA(COTS)NRO,Success[9],F9 v1.0[7]B0004.1[8],Failure[9][14](parachute),8 December 2010,15:43
2,3,"CCAFS,SLC-40",Dragondemo flight C2+[18](Dragon C102),"525kg (1,157lb)[19]",LEO(ISS),NASA(COTS),Success[20],F9 v1.0[7]B0005.1[8],Noattempt,22 May 2012,07:44
3,4,"CCAFS,SLC-40",SpaceX CRS-1[22](Dragon C103),"4,700kg (10,400lb)",LEO(ISS),NASA(CRS),Success,F9 v1.0[7]B0006.1[8],No attempt,8 October 2012,00:35
4,5,"CCAFS,SLC-40",SpaceX CRS-2[22](Dragon C104),"4,877kg (10,752lb)",LEO(ISS),NASA(CRS),Success,F9 v1.0[7]B0007.1[8],Noattempt,1 March 2013,15:10


**Explanation:**
- `pd.Series(value)` converts each list into a pandas Series (a column).
- `if len(value) > 0` skips any keys that were never filled in.
- `pd.DataFrame({...})` combines all these Series into a single table.

In [ ]:
df2.to_csv('dataset_part_2.csv', index=False)
print("Saved: dataset_part_2.csv")

Saved: dataset_part_2.csv
